In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

In [2]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location=('/workspaces/MLOps-ZoomCamp/02-Experiment tracking and model '
 'management/mlruns/1'), creation_time=1748500545859, experiment_id='1', last_update_time=1748500545859, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

# 01.data preprocessing 

In [3]:
def read_dataframe(filename):
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)

        df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
        df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    elif filename.endswith(".parquet"):
        df = pd.read_parquet(filename)

    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

In [4]:
df_train = read_dataframe("../data/yellow_tripdata_2023-01.parquet")
df_val = read_dataframe("../data/yellow_tripdata_2023-02.parquet")

In [5]:
categorical = ["PULocationID", "DOLocationID"]

# Turn dataframes into list of dictionaries
train_dicts = df_train[categorical].to_dict(orient="records")
val_dicts = df_val[categorical].to_dict(orient="records")

In [6]:
# Fit dictionary vectorizer on Training data
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

# Apply learned dictionary vectorizer on validation data
X_val = dv.transform(val_dicts)

In [7]:
# Set GT Values
y_train = df_train["duration"].values
y_val = df_val["duration"].values

# 02. trying out Linear Regression without logging

In [8]:
# Train linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [9]:
# Make predictions on validation data
y_pred = lr.predict(X_val)

# Calculate RMSE on validation
rmse = root_mean_squared_error(y_val, y_pred)

In [10]:
rmse

7.811817745843695

# 03.trying out Lasso with simple logging

In [11]:
# trying and logging lasso
with mlflow.start_run():
    mlflow.set_tag("developer", "Wahba")

    mlflow.log_param("train-data-path", "../data/yellow_tripdata_2023-01.parquet")
    mlflow.log_param("valid-data-path", "../data/yellow_tripdata_2023-02.parquet")

    alpha = 0.001
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

# 04.Hyperparameter Optimization for XGBoost Using Hyperopt and MLflow

This section performs hyperparameter tuning for an XGBoost model using Hyperopt, with each trial's parameters and RMSE logged to MLflow for experiment tracking.

**The best hyperparameters identified will be used below for final model training.**


In [12]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [13]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50,
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {"loss": rmse, "status": STATUS_OK}


search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
    "seed": 42,
}
## skipping it as i already ran it for hours XD
# best_result = fmin(
#     fn=objective, space=search_space, algo=tpe.suggest, max_evals=50, trials=Trials()
# )

## Training XGBoost Model with Manual Hyperparameters and Logging to MLflow

In [14]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    params = {
        "learning_rate": 0.37423890981602775,
        "max_depth": 30,
        "min_child_weight": 2.4679201221277838,
        "objective": "reg:squarederror",
        "reg_alpha": 0.008474358456426086,
        "reg_lambda": 0.006361637376418637,
        "seed": 42,
        "eval_metric": "rmse",
    }

    # autolog for xgboost (disabled it, kernel crashing in GH codesapces)
    # mlflow.xgboost.autolog(disable=True)

    # manual Log parameters
    mlflow.log_params(params)

    booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50,
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # log preprocessor (the dict-vectorizer)
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models")


[0]	validation-rmse:8.65467
[1]	validation-rmse:7.95284
[2]	validation-rmse:7.53226
[3]	validation-rmse:7.35365
[4]	validation-rmse:7.20355
[5]	validation-rmse:7.09495
[6]	validation-rmse:6.95770
[7]	validation-rmse:6.63738
[8]	validation-rmse:6.59351
[9]	validation-rmse:6.56125
[10]	validation-rmse:6.52747
[11]	validation-rmse:6.39761
[12]	validation-rmse:6.36861
[13]	validation-rmse:6.26318
[14]	validation-rmse:6.24791
[15]	validation-rmse:6.23092
[16]	validation-rmse:6.21238
[17]	validation-rmse:6.19230
[18]	validation-rmse:6.10277
[19]	validation-rmse:6.07290
[20]	validation-rmse:6.06297
[21]	validation-rmse:6.05279
[22]	validation-rmse:6.04313
[23]	validation-rmse:5.96189
[24]	validation-rmse:5.90946
[25]	validation-rmse:5.88595
[26]	validation-rmse:5.87381
[27]	validation-rmse:5.79263
[28]	validation-rmse:5.71704
[29]	validation-rmse:5.71066
[30]	validation-rmse:5.70618
[31]	validation-rmse:5.68750
[32]	validation-rmse:5.68213
[33]	validation-rmse:5.67701
[34]	validation-rmse:5.6

/home/codespace/anaconda3/envs/mlflow_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:23:09] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/05/30 06:23:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
